# 01. Planner-Worker-Critic

三 Agent 协作模式是最经典的多 Agent 架构：

- **Planner**：接收高层目标，分解为子任务，分配给 Worker
- **Worker**：执行分配的任务，返回结果
- **Critic**：检查输出是否满足标准，决定通过还是打回

> **检查点**：能画出 Planner → Worker → Critic 的数据流和反馈回路。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'agents'))

from scripts.nb_helpers import setup, chat, chat_json, chat_with_tools, show_trace
client = setup()


🌐 API 模式 — model=deepseek-flash  base_url=https://api.deepseek.com


In [2]:
from dataclasses import dataclass, field

@dataclass
class SubTask:
    id: str; desc: str; assignee: str; status: str = "pending"
    result: str = ""; feedback: str = ""

class Planner:
    def decompose(self, goal: str) -> list[SubTask]:
        if "report" in goal.lower():
            return [
                SubTask("S1","Gather data","Worker-A"),
                SubTask("S2","Analyze data","Worker-B",result="depends on S1"),
                SubTask("S3","Write report","Worker-A"),
            ]
        return [SubTask("S1",f"Execute: {goal}","Worker-A")]

class Worker:
    def execute(self, task: SubTask) -> str:
        results = {"Gather data":"Collected 150 data points",
                   "Analyze data":"Mean=42.3, std=8.1",
                   "Write report":"# Report\n\nBased on analysis..."}
        return results.get(task.desc, f"Completed: {task.desc}")

class Critic:
    def evaluate(self, task: SubTask) -> tuple[bool, str]:
        if "data" in task.desc.lower() and "collected" not in task.result.lower():
            return False, "Needs specific numbers, not generic description"
        if len(task.result) < 20:
            return False, "Output too short, expected detailed response"
        return True, "Meets criteria"

# 运行流水线
planner = Planner(); worker = Worker(); critic = Critic()
tasks = planner.decompose("write climate report")

for task in tasks:
    print(f"\n[{task.id}] {task.desc} → {task.assignee}")
    task.result = worker.execute(task)
    print(f"  Result: {task.result[:60]}...")
    ok, feedback = critic.evaluate(task)
    task.feedback = feedback
    task.status = "done" if ok else "revise"
    print(f"  Critic: {'✓' if ok else '✗'} {feedback}")


[S1] Gather data → Worker-A
  Result: Collected 150 data points...
  Critic: ✓ Meets criteria

[S2] Analyze data → Worker-B
  Result: Mean=42.3, std=8.1...
  Critic: ✗ Needs specific numbers, not generic description

[S3] Write report → Worker-A
  Result: # Report

Based on analysis......
  Critic: ✓ Meets criteria


## 反馈回路设计

Critic 不只是判断「过/不过」，还需要给出可操作的反馈：
- 差：「This is wrong」
- 好：「Missing temperature data for 2020-2023. Add from NOAA dataset.」

**好反馈的特征**：具体、可操作、引用证据。

> **检查点**：能写出一个 Critic 的评估 rubric（评分表），包含至少 3 个维度。

## 真实 API 实验

跑真实 LLM 驱动的 Planner-Worker 团队（`role_team_demo.py --use-api`）。


In [3]:
SCRIPT = ROOT / 'agents' / 'orchestration' / 'scripts' / 'role_team_demo.py'
import subprocess, sys
result = subprocess.run([sys.executable, str(SCRIPT), '--use-api', '--verbose'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr, file=sys.stderr)


Scenario: climate impacts [LLM]

[Researcher] **Climate impacts (per IPCC AR6 and NOAA/WMO assessments):** Human-caused warmin...

[Writer] Human-caused warming of ~1.1 °C (2011–2020 vs. 1850–1900) is already intensifying heatwaves, heavy precipitation, and re...

[FactChecker] Verified: the stated warming (~1.1 °C), sea-level rise (~0.20 m since 1901), and observed impacts are consistent with IPCC AR6 and NOAA/WMO assessments. For a focused synthesis, agriculture illustrates the risks: warming has slowed global agricultural productivity growth and reduced yields of some staple crops in lower-latitude regions, while heat, drought, and heavy precipitation increase food insecurity. CO₂ fertilization offers only a partial, regionally variable offset and can reduce crop nutrient density, so impacts worsen with higher warming levels.




## 练习

1. 给 Critic 增加评分维度：正确性（事实准确）、完整性（覆盖需求）、清晰度（可读性）。用加权评分计算最终结果。
2. 实现 Worker 的「收到反馈 → 修改 → 重新提交」循环，最多 3 轮。
3. 对比 Planner-Worker-Critic 与单 Agent loop 的优缺点。在什么情况下多 Agent 反而不如单 Agent？

> **检查点**：能为你的业务场景实现 Planner-Worker-Critic 流水线，并正确处理反馈循环。